In [ ]:
import pandas as pd
import numpy as np
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.metrics import accuracy_score, roc_auc_score, roc_curve, auc, precision_score, recall_score, f1_score
from sklearn.dummy import DummyClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, StackingClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import Pipeline
import matplotlib.pyplot as plt
import joblib

In [ ]:
df = pd.read_csv("S06-hw-dataset-01.csv")
df.head()

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
df["target"].describe()

In [ ]:
X = df.drop(columns=["id", "target"])
y = df["target"].to_frame()

Фиксированный seed важен для воспроизводимости результатов, так как он гарантирует, что при каждом запуске кода данные разделяются одинаково, что позволяет корректно сравнивать разные модели и отлаживать код. Стратификация критична для сохранения распределения классов, потому что она предотвращает ситуацию, когда в обучающей или тестовой выборке оказывается непропорционально мало примеров минорного класса, что особенно важно при дисбалансе данных



In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

In [ ]:
dc = DummyClassifier(strategy="most_frequent")
dc.fit(X_train, y_train)
dc = DummyClassifier(strategy="most_frequent", random_state=42)
dc.fit(X_train,y_train)

y_pred = dc.predict(X_test)
f1 = f1_score(y_test, y_pred)
accuracy = accuracy_score(y_test, y_pred)
y_pred_rac = roc_auc_score(y_test, dc.predict_proba(X_test)[:, 1])
print("f1 =", f1)
print("accuracy =", accuracy)
print("roc_auc_store =", y_pred_rac)

In [ ]:
logreg_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('logreg', LogisticRegression(random_state=42, max_iter=1000))
])

logreg_pipeline.fit(X_train, y_train)

y_pred_lr = logreg_pipeline.predict(X_test)
y_proba_lr = logreg_pipeline.predict_proba(X_test)


f1_lr = f1_score(y_test, y_pred_lr, average='macro')  # или 'weighted', 'micro'
accuracy_lr = accuracy_score(y_test, y_pred_lr)
roc_auc_lr = roc_auc_score(y_test, y_proba_lr, multi_class='ovr', average='macro')

print("F1-score (macro):", f1_lr)
print("Accuracy:", accuracy_lr)
print("ROC-AUC (ovr, macro):", roc_auc_lr)


In [ ]:
param_grid_dt = {
    'max_depth': [3, 5, 7, 10, 15, None],
    'min_samples_leaf': [1, 2, 3, 4, 5, 10],
    'min_samples_split': [2, 5, 10, 20],
    'criterion': ['gini', 'entropy']
}

dt = DecisionTreeClassifier(random_state=42)

grid_search_dt = GridSearchCV(
    dt,
    param_grid_dt,
    cv=5,
    scoring='f1_macro',
    n_jobs=-1,
    verbose=1
)

grid_search_dt.fit(X_train, y_train)

print(f"Лучшие параметры: {grid_search_dt.best_params_}")
print(f"Лучший CV score: {grid_search_dt.best_score_:.4f}")

best_dt = grid_search_dt.best_estimator_
y_pred_dt = best_dt.predict(X_test)
y_proba_dt = best_dt.predict_proba(X_test)

f1_dt = f1_score(y_test, y_pred_dt, average='macro')
accuracy_dt = accuracy_score(y_test, y_pred_dt)
roc_auc_dt = roc_auc_score(y_test, y_proba_dt, multi_class='ovr', average='macro')

print(f"\nРезультаты на тесте:")
print(f"F1-score (macro): {f1_dt:.4f}")
print(f"Accuracy: {accuracy_dt:.4f}")
print(f"ROC-AUC: {roc_auc_dt:.4f}")

In [ ]:
param_grid_rf = {
    'n_estimators': [50, 100, 200],
    'max_depth': [10, 20, 30, None],
    'min_samples_leaf': [1, 2, 4],
    'max_features': ['sqrt', 'log2', 0.5]
}

rf = RandomForestClassifier(random_state=42, n_jobs=-1)

grid_search_rf = GridSearchCV(
    rf,
    param_grid_rf,
    cv=3,
    scoring='f1_macro',
    n_jobs=-1,
    verbose=1
)

grid_search_rf.fit(X_train, y_train)

print(f"Лучшие параметры: {grid_search_rf.best_params_}")
print(f"Лучший CV score: {grid_search_rf.best_score_:.4f}")

best_rf = grid_search_rf.best_estimator_
y_pred_rf = best_rf.predict(X_test)
y_proba_rf = best_rf.predict_proba(X_test)

f1_rf = f1_score(y_test, y_pred_rf, average='macro')
accuracy_rf = accuracy_score(y_test, y_pred_rf)
roc_auc_rf = roc_auc_score(y_test, y_proba_rf, multi_class='ovr', average='macro')

print(f"\nРезультаты на тесте:")
print(f"F1-score (macro): {f1_rf:.4f}")
print(f"Accuracy: {accuracy_rf:.4f}")
print(f"ROC-AUC: {roc_auc_rf:.4f}")

Fitting 3 folds for each of 108 candidates, totalling 324 fits


In [ ]:
param_grid_gb = {
    'n_estimators': [50, 100],
    'learning_rate': [0.01, 0.1, 0.2],
    'max_depth': [3, 5, 7],
    'min_samples_leaf': [1, 2, 4]
}

gb = GradientBoostingClassifier(random_state=42)


grid_search_gb = GridSearchCV(
    gb,
    param_grid_gb,
    cv=3,
    scoring='f1_macro',
    n_jobs=-1,
    verbose=1
)

grid_search_gb.fit(X_train, y_train)

print(f"Лучшие параметры: {grid_search_gb.best_params_}")
print(f"Лучший CV score: {grid_search_gb.best_score_:.4f}")

best_gb = grid_search_gb.best_estimator_
y_pred_gb = best_gb.predict(X_test)
y_proba_gb = best_gb.predict_proba(X_test)

f1_gb = f1_score(y_test, y_pred_gb, average='macro')
accuracy_gb = accuracy_score(y_test, y_pred_gb)
roc_auc_gb = roc_auc_score(y_test, y_proba_gb, multi_class='ovr', average='macro')

print(f"\nРезультаты на тесте:")
print(f"F1-score (macro): {f1_gb:.4f}")
print(f"Accuracy: {accuracy_gb:.4f}")
print(f"ROC-AUC: {roc_auc_gb:.4f}")

In [ ]:
base_models = [
    ('dt', DecisionTreeClassifier(max_depth=5, min_samples_leaf=4, random_state=42)),
    ('rf', RandomForestClassifier(n_estimators=50, max_depth=10, random_state=42, n_jobs=-1)),
    ('gb', GradientBoostingClassifier(n_estimators=50, learning_rate=0.1, max_depth=3, random_state=42))
]


meta_model = LogisticRegression(random_state=42, max_iter=1000)


stacking_clf = StackingClassifier(
    estimators=base_models,
    final_estimator=meta_model,
    cv=5,
    n_jobs=-1
)


stacking_clf.fit(X_train, y_train)


y_pred_stack = stacking_clf.predict(X_test)
y_proba_stack = stacking_clf.predict_proba(X_test)

f1_stack = f1_score(y_test, y_pred_stack, average='macro')
accuracy_stack = accuracy_score(y_test, y_pred_stack)
roc_auc_stack = roc_auc_score(y_test, y_proba_stack, multi_class='ovr', average='macro')

print(f"\nРезультаты на тесте:")
print(f"F1-score (macro): {f1_stack:.4f}")
print(f"Accuracy: {accuracy_stack:.4f}")
print(f"ROC-AUC: {roc_auc_stack:.4f}")

In [ ]:
results = []


models_data = [
    ('DecisionTree', f1_dt, accuracy_dt, roc_auc_dt),
    ('RandomForest', f1_rf, accuracy_rf, roc_auc_rf),
    ('GradientBoosting', f1_gb, accuracy_gb, roc_auc_gb),
]

if 'f1_stack' in locals():
    models_data.append(('Stacking', f1_stack, accuracy_stack, roc_auc_stack))

for name, f1, acc, roc in models_data:
    results.append({
        'Model': name,
        'F1-score': f1,
        'Accuracy': acc,
        'ROC-AUC': roc
    })

results_df = pd.DataFrame(results)
results_df = results_df.sort_values('F1-score', ascending=False)
print(results_df.to_string(index=False))

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for i, metric in enumerate(['F1-score', 'Accuracy', 'ROC-AUC']):
    axes[i].bar(results_df['Model'], results_df[metric])
    axes[i].set_title(f'Сравнение {metric}')
    axes[i].set_ylabel(metric)
    axes[i].tick_params(axis='x', rotation=45)
    axes[i].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

best_model = results_df.iloc[0]
print("Лучшая модель: {best_model['Model']}")
print(f"F1-score: {best_model['F1-score']:.4f}")
print(f"Accuracy: {best_model['Accuracy']:.4f}")
print(f"ROC-AUC: {best_model['ROC-AUC']:.4f}")